# Photometry notebook

Assumes aggregated DAta has been produced

In [3]:
import sys
from pathlib import Path

# Use __file__ if it exists (script), otherwise use the current working directory (notebook)
try:
    THIS_FILE = Path(__file__).resolve()
except NameError:
    THIS_FILE = Path.cwd() / "DA.ipynb"

PROJECT_ROOT = THIS_FILE.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root on sys.path:", PROJECT_ROOT)


import os
import glob
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import zscore
from scipy.signal import savgol_filter
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.ndimage import gaussian_filter1d

from ratcode.config.paths import PATH_STORE_PICKLES, DROPBOX_TASK_PATH, PATH_DATAFRAMES
from ratcode.common.logging import determine_experiment
from ratcode.common.colorcodes import *
from ratcode.behavior import change_point
from ratcode.photometry.photometry import get_prediction, quantile_regression, signal2eventsnippets, find_poly, segment_and_fit_function, butter_filter, mask_jumps, make_continuous, compute_snippets_across_days, bootstrap_ci, categorize_presses, envelope_quantile_regression
from ratcode.common.dataframe import group_and_listify
from ratcode.common.time import convert_date_bonsai, convert_timestamp
from ratcode.common.math import drop_nans_matrix
from ratcode.common.colorcodes import FI_order, color_FI_blocks, rwd_order, color_rwd_blocks

from ratcode.init import setup

setup()

Project root on sys.path: d:\Learning Lab Dropbox\Learning Lab Team Folder\Patlab protocols\Code\Python\SF\rats_ficlickrwd


In [4]:
aggregated_jointdf = pd.read_pickle(rf'{PATH_DATAFRAMES}\aggregate_photometry_Palladium_Ruthenium.pkl')


using code from the aggregate_photometry (thesis nb), which will eventually be deleted

In [ ]:
# knobs
alignment_idx = 1
baseline_correct = False
DA_column = 'DA_zscored_session'
#DA_column = 'DA_envelope_z'

animal = 'Palladium'
df = aggregated_jointdf.query(f'animal == "{animal}"')# and date in {photometry_dates_dict[animal]}')

#df = allphotometrydf.query(f'animal == "{animal}"')


window = (-4,4)
zero_time = int((window[1] - window[0])/2*100)

alignments = ['cp_abs', 'last_lever_abs']
alignment_labels = ['transition point', 'last press']

if alignment_idx == 1:
    baseline_start = zero_time-30
    baseline_end = zero_time
else:
    baseline_start = zero_time-100
    baseline_end = zero_time-50



baseline_title = ' | baseline corrected' if baseline_correct else ''

exp = 'a'
snippets_a = []
for FI in FI_order:
    time, snippets = compute_snippets_across_days(df,
        f'experiment == "{exp}" and FI == {FI}', alignments[alignment_idx], DA_column, window)
    snippets_a.append(snippets)

exp = 'b'
snippets_b = []
for FI in FI_order:
    time, snippets = compute_snippets_across_days(df,
        f'experiment == "{exp}" and FI == {FI}', alignments[alignment_idx], DA_column, window)
    snippets_b.append(snippets)

exp = 'c'
snippets_c = []
for nprots in rwd_order:
    time, snippets = compute_snippets_across_days(df, 
        f'experiment == "{exp}" and n_protocols == {nprots}', alignments[alignment_idx], DA_column, window)
    snippets_c.append(snippets)



fig, axs = plt.subplots(1,3, figsize = (12,4), tight_layout = True, sharex = True, sharey = True)

for ii in range(3):
    if baseline_correct:
        axs[0].plot(time,np.nanmean(snippets_a[ii], axis = 0) - np.mean(np.nanmean(snippets_a[ii], axis=0)[baseline_start:baseline_end]), color = color_FI_blocks[ii])
        axs[1].plot(time,np.nanmean(snippets_b[ii], axis = 0) - np.mean(np.nanmean(snippets_b[ii], axis=0)[baseline_start:baseline_end]), color = color_FI_blocks[ii])
        axs[2].plot(time,np.nanmean(snippets_c[ii], axis = 0) - np.mean(np.nanmean(snippets_c[ii], axis=0)[baseline_start:baseline_end]), color = color_nprots_blocks[ii])
            
    else:
        axs[0].plot(time,np.nanmean(snippets_a[ii], axis = 0), color = color_FI_blocks[ii])
        axs[1].plot(time,np.nanmean(snippets_b[ii], axis = 0), color = color_FI_blocks[ii])
        axs[2].plot(time,np.nanmean(snippets_c[ii], axis = 0), color = color_rwd_blocks[ii])

    axs[ii].axvline(0, color = 'grey', lw = 1)

    axs[ii].set_xlabel(f'time since {alignment_labels[alignment_idx]} (s)')

axs[0].set_title('varying FI')
axs[1].set_title('fixed reward rate')
axs[2].set_title('varying reward magnitude')

axs[0].set_ylabel('DA (z ΔF/F)')

figtitle = f'{animal} all | DA aligned to {alignment_labels[alignment_idx]}{baseline_title} | {DA_column}'
#figtitle = f'{animal} all | DA aligned to {alignment_labels[alignment_idx]}{baseline_title}_RPE'
fig.suptitle(figtitle)
fig.savefig(fr'{PATH_SAVE_AGGREGATE_DA_FIGS}\{figtitle.replace('|', '_')}.png')
#fig.savefig(fr'{photometry_fig_path}\{figtitle.replace('|', '_')}.pdf')
